# S6E6 — Second Phase (Project 기반)

이전 판과 달라진 점만 먼저:

| 예전 | 지금 |
|---|---|
| `PipelineBuilder(path='exp', name=...)` | `project.pipeline_builder(name)` |
| `p.set_grp(..., role='stage'/'head')` | `role` 없음 — Pipeline은 **노드 전용** |
| 모델을 `set_node(grp='xgb')`로 선언 | 모델은 **`Trial`** — Pipeline 밖, `exp()`에 직접 전달 |
| `p.build()` | `project.build_pipeline(p)` → 버전 부여 후 `e.set_pipeline(...)` |
| `Experimenter(df, sp=, path=)` | `project.experimenter(name, df, ...)` / `project.load_experimenter(name, df)` |
| `e.set_collector(...)` / `e.get_collector(...)` | `e.collectors` 레지스트리 (run 소유, 등록 즉시 영속화) |
| `e.exp(nodes='xgb1')` | `e.exp([(trial, outer, inner), ...], project.trials, collectors=[이름, ...])` |
| `e.exp(finalize=True)` | `finalize` 개념 없음 |
| 수집 실패는 `collector.warnings` (메모리) | `e.collectors.hist` — **`CollectHist`** (fold 단위 이력) |

grp 상속이 모델에 적용되지 않으므로, 그 자리는 아래 `MODELS` dict + `trial()` 헬퍼가 대신한다.

In [1]:
import os
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import mllabs

data_path = Path('data')

In [2]:
from mllabs.processor import PolarsLoader
ploader = PolarsLoader(predefined_types={'id': pl.Int64, 'obj_id': pl.Float64}, infer_schema_length=10000)
ploader.fit([data_path / 'train.csv', data_path / 'test.csv', data_path / 'star_classification.csv'])
df_train = ploader.transform([data_path / 'train.csv'])
df_test = ploader.transform([data_path / 'test.csv']).with_columns(
   pl.lit('').alias('class')
)

In [3]:
from itertools import combinations
from mllabs.processor import ExprProcessor
expr_dict1 = {
    'u': pl.when(pl.col('u') < 10).then(
        pl.when(pl.col("class") == "STAR").then(pl.col("u")).otherwise(None).mean()
    ).otherwise(pl.col('u')),
    'alpha90': pl.col('alpha') + 90,
    'spectral_type_galaxy_population': (pl.col('spectral_type').cast(pl.String) + '_' + pl.col('galaxy_population').cast(pl.String)).cast(pl.Categorical)
}
X_mags = ['u', 'g', 'r', 'i', 'z']
expr_dict2 = {
    'mag_mean': pl.mean_horizontal(*X_mags),
    'mag_std': pl.concat_list(X_mags).list.std(),
    'mag_min': pl.min_horizontal(*X_mags),
    'mag_max': pl.max_horizontal(*X_mags),
    'mag_range': pl.max_horizontal(*X_mags) - pl.min_horizontal(*X_mags),
}
X_mags_stat = list(expr_dict2.keys())
expr_dict2 = {
    **expr_dict2,
    'mag_vmax': pl.struct(X_mags).map_elements(
        lambda x: max(x, key=x.get),
        return_dtype=pl.String),
    'redshift_log': (pl.col('redshift') + 1e-1).log(),
    'redshift_1e-4': (pl.col('redshift') == 0.0001).cast(pl.Int8),
    'spectral_type_ord': pl.col('spectral_type').replace({'M': 0, 'G/K': 1, 'A/F': 2, 'O/B': 3}).to_physical().cast(pl.Int8),
    'galaxy_population_i': pl.when(pl.col('galaxy_population') == 'Red_Sequence').then(1).otherwise(0).cast(pl.Int8),
}
X_diff = list()
for i, j in combinations(X_mags, 2):
    X_diff.append(f'{i}_{j}')
    expr_dict2[X_diff[-1]] = pl.col(i) - pl.col(j)

X_mags_log = list()
for i in X_mags:
    X_mags_log.append(f'{i}_log')
    expr_dict2[X_mags_log[-1]] = pl.col(i).log()

In [4]:
from sklearn.pipeline import make_pipeline
expr_p = make_pipeline(ExprProcessor(expr_dict1), ExprProcessor(expr_dict2))
df_train = expr_p.fit_transform(df_train)
df_test = expr_p.transform(df_test)

In [5]:
import pickle as pkl
if not os.path.exists('data/lof.pkl'):
    from sklearn.neighbors import LocalOutlierFactor
    from sklearn.cluster import KMeans
    df_lof = pl.concat([df_train[['alpha90', 'delta']], df_test[['alpha90', 'delta']]])
    lof = LocalOutlierFactor()
    lof.fit(df_lof)
    lof_ = lof.negative_outlier_factor_
    clu_kmeans = KMeans(3000)
    clu_kmeans.fit(df_lof)
    km3000 = clu_kmeans.labels_
    with open('data/lof.pkl', 'wb') as f:
        pkl.dump((lof_, km3000), f)
else:
    with open('data/lof.pkl', 'rb') as f:
        lof_, km3000 = pkl.load(f)

df_train = df_train.with_columns(
    pl.Series('lof', lof_[:len(df_train)]),
    pl.Series('km3000', km3000[:len(df_train)], dtype=pl.String).cast(pl.Categorical)
)
df_test = df_test.with_columns(
    pl.Series('lof', lof_[len(df_train):]),
    pl.Series('km3000', km3000[len(df_train):], dtype=pl.String).cast(pl.Categorical)
)
df_train.head()

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class,alpha90,spectral_type_galaxy_population,mag_mean,mag_std,mag_min,mag_max,mag_range,mag_vmax,redshift_log,redshift_1e-4,spectral_type_ord,galaxy_population_i,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,u_log,g_log,r_log,i_log,z_log,lof,km3000
i64,f32,f32,f32,f32,f32,f32,f32,f32,cat,cat,cat,f32,cat,f32,f32,f32,f32,f32,str,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,cat
0,147.734253,16.959272,25.472122,21.895559,20.357925,19.257113,18.621058,0.408982,"""M""","""Red_Sequence""","""GALAXY""",237.734253,"""M_Red_Sequence""",21.120754,2.731221,18.621058,25.472122,6.851065,"""u""",-0.675342,0,0,1,3.576563,5.114197,6.21501,6.851065,1.537634,2.638447,3.274502,1.100813,1.736868,0.636055,3.237585,3.086284,3.01347,2.957881,2.924293,-1.265492,"""1763"""
1,127.988678,32.346718,20.778509,19.087063,17.587208,17.226067,16.786432,0.157976,"""M""","""Red_Sequence""","""GALAXY""",217.988678,"""M_Red_Sequence""",18.293055,1.636653,16.786432,20.778509,3.992077,"""u""",-1.35489,0,0,1,1.691446,3.191301,3.552443,3.992077,1.499855,1.860996,2.300631,0.361141,0.800776,0.439634,3.03392,2.949011,2.867172,2.846424,2.820571,-1.085606,"""1223"""
2,179.792648,35.344845,21.035202,21.079128,21.171841,20.58263,20.557365,2.82377,"""O/B""","""Blue_Cloud""","""QSO""",269.792664,"""O/B_Blue_Cloud""",20.885233,0.292103,20.557365,21.171841,0.614475,"""r""",1.072874,0,3,0,-0.043926,-0.136639,0.452572,0.477837,-0.092712,0.496498,0.521763,0.589211,0.614475,0.025265,3.046198,3.048284,3.052672,3.024448,3.02322,-1.186385,"""619"""
3,225.818298,48.56942,23.305056,21.050735,19.017754,18.365658,17.914951,0.536099,"""M""","""Red_Sequence""","""GALAXY""",315.818298,"""M_Red_Sequence""",19.93083,2.235331,17.914951,23.305056,5.390104,"""u""",-0.452402,0,0,1,2.25432,4.287302,4.939398,5.390104,2.032982,2.685078,3.135784,0.652096,1.102802,0.450706,3.148671,3.046936,2.945373,2.910483,2.885636,-0.991541,"""1492"""
4,141.836136,19.342852,21.703157,19.47168,18.234449,17.899446,17.616184,0.555761,"""M""","""Red_Sequence""","""GALAXY""",231.836136,"""M_Red_Sequence""",18.984983,1.676354,17.616184,21.703157,4.086973,"""u""",-0.421958,0,0,1,2.231478,3.468708,3.803711,4.086973,1.23723,1.572233,1.855495,0.335003,0.618265,0.283262,3.077458,2.968961,2.903313,2.88477,2.868818,-1.048082,"""1279"""


In [6]:
y2 = 'class'
class_weight = df_train[y2].to_pandas().value_counts().pipe(
    lambda x: x / x.min()
).to_dict()
class_weight

{'GALAXY': 4.56312557419854, 'QSO': 1.4160703060780426, 'STAR': 1.0}

In [7]:
y = 'class_i'
y_repl = {'STAR': 0, 'QSO': 1, 'GALAXY': 2}
df_train = df_train.with_columns(
    **{
        y: pl.col(y2).replace(y_repl).cast(pl.Int8),
        'sample_weight': pl.col(y2).cast(pl.String).replace(class_weight).cast(pl.Float32)
    }
)

In [8]:
X_loc = ['alpha', 'delta']
X_num = ['redshift', 'redshift_log', 'lof', 'alpha90']
X_bin = ['redshift_1e-4', 'galaxy_population_i']
X_nom = ['galaxy_population', 'spectral_type', 'spectral_type_galaxy_population']
X_base = X_loc[1:] + X_num + X_bin + X_nom[1:] + X_diff + X_mags_stat + X_mags_log

X_nom2 = ['km3000']
X_ohe = ['spectral_type', 'spectral_type_galaxy_population']
X_std = ['redshift_log', 'lof'] + X_mags + X_mags_stat + X_mags_log + X_diff

X_num = X_loc + X_num + X_mags + X_mags_stat + X_mags_log + X_diff
X_nom = X_nom + X_nom2

## Project 구성

`Project`가 디렉토리 레이아웃과 프로젝트 전역인 것들(pipelines / TrialStore / cache)을 소유한다.
run 하나(Experimenter)에 관한 것 — splitter, 채택한 Pipeline, 노드 아티팩트, Collector — 은 전부
`exp/phase2/` 안에 있어서 Project 없이도 열 수 있다.

In [9]:
# 처음부터 다시 돌릴 때만
# !rm -rf exp

In [10]:
from mllabs import Project, Trial
from mllabs import ProgressSessionLogger, TqdmProgressSession
from sklearn.model_selection import StratifiedShuffleSplit
from IPython.display import display, Markdown

logger = ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)

project = Project('exp')
p = project.pipeline_builder('phase2_pipeline')

In [11]:
p.set_datasource({
    **{i: 'numerical' for i in X_num},
    **{i: 'binary' for i in X_bin},
    **{i: 'nominal' for i in X_nom + [y, y2]},
}, targets=[y, y2])

'skip'

### Pipeline — 전처리 노드만

`role`이 없어진 뒤로 Pipeline에 담기는 건 전처리 노드뿐이다. 예전에 `role='head'`로 선언하던 모델 그룹
(`clf`/`xgb`/`lgb`/...)은 여기 들어가지 않는다.

### 모델 — Trial

`Trial`은 Pipeline 밖이라 **grp 상속이 없다.** 예전에 `set_grp('xgb', parent='clf', ...)`가 해주던
공통 파라미터 상속은 아래 `MODELS` dict와 `trial()` 헬퍼가 대신한다 — 어차피 파이썬 dict 병합이라
상속 규칙을 따로 외울 필요가 없어졌다.

`edges_for()`는 라운드마다 namespace 세그먼트(`tgt_delta:(*)` 등)를 덧붙이는 부분을 한곳에 모은 것이다.

In [12]:
def dsl_set(cols):
    return '{' + ', '.join(cols) + '}'


y_edges = {'y': dsl_set([y])}

p.set_grp('pre', method='transform')
p.set_grp('pre_ft', method='fit_transform', edges=y_edges)
p.set_grp('pre_ft2', method='fit_transform', edges={'y': dsl_set([y2])})

p.set_node('std', grp='pre', processor='sklearn.preprocessing.StandardScaler', edges={'X': dsl_set(X_std)})
p.set_node('ohe', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': dsl_set(X_ohe)}, params={'sparse_output': False})
p.set_node('coov', grp='pre', processor='mllabs.processor.CatOOVFilter', edges={'X': dsl_set(X_nom)})
p.set_node('tgt_km3000', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': dsl_set(X_nom2)}, params={'target_type': 'multiclass'})

{'result': 'skip',
 'affected_nodes': [],
 'old_obj': <mllabs._pipeline._PipelineNode at 0x7e78e999c7a0>,
 'obj': <mllabs._pipeline._PipelineNode at 0x7e78e999c7a0>}

In [13]:
MODELS = {
    'xgb': dict(
        processor='xgboost.XGBClassifier',
        adapter={'__ref__': 'mllabs.adapter.XGBoostAdapter', '__params__': {'eval_mode': 'both'}},
        params={'random_state': 123, 'n_estimators': 10000, 'enable_categorical': True,
                'early_stopping_rounds': 50, 'eval_metric': 'mlogloss'},
    ),
    'lgb': dict(
        processor='lightgbm.LGBMClassifier',
        adapter={'__ref__': 'mllabs.adapter.LightGBMAdapter', '__params__': {'eval_mode': 'both'}},
        params={'random_state': 123, 'n_estimators': 10000, 'verbose': -1, 'gpu': None,
                'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},
                'eval_metric': 'multi_logloss'},
    ),
    'cb': dict(
        processor='catboost.CatBoostClassifier',
        adapter={'__ref__': 'mllabs.adapter.CatBoostAdapter', '__params__': {'eval_mode': 'valid'}},
        params={'random_state': 123, 'n_estimators': 10000, 'early_stopping_rounds': 50,
                'eval_metric': 'AUC', 'verbose': 0,
                'cat_features': {'__ref__': 'mllabs.ColSelector',
                                 '__params__': {'dsl_string': '*@categorical'}}},
    ),
    'nn': dict(
        processor='mllabs.nn.NNClassifier',
        params={'metrics': ['sparse_categorical_crossentropy'], 'early_stopping': 10, 'epochs': 200},
    ),
    'lr': dict(processor='sklearn.linear_model.LogisticRegression', params={}),
    'dt': dict(processor='sklearn.tree.DecisionTreeClassifier', params={'random_state': 123}),
}


def trial(name, model, X, **params):
    m = MODELS[model]
    return Trial(name, m['processor'], {'X': X, **y_edges}, method='predict',
                 adapter=m.get('adapter'), params={**m['params'], **params})


def edges_for(*extra, xgb_cols=None):
    ns = ''.join(f' + {n}:(*)' for n in extra)
    linear = 'std:(*) + ohe:(*@ohe_drop_first)'
    return {
        'xgb': dsl_set(xgb_cols if xgb_cols is not None else X_num + X_bin) + ' + coov:(*)' + ns,
        'lgb': dsl_set(X_base) + ns,
        'cb': dsl_set(X_base) + ns,
        'nn': linear + ns,
        'lr': linear + ns,
    }


def round_trials(idx, *extra, **kw):
    E = edges_for(*extra, **kw)
    return [trial(f'{m}{idx}', m, E[m]) for m in ('xgb', 'lgb', 'cb', 'nn', 'lr')]

In [15]:
def adopt():
    """현재 builder 상태를 새 버전으로 빌드해 이 run에 채택시킨다.
    바뀐 노드만 stale 처리되어 아티팩트가 지워진다 (Pipeline.diff_from)."""
    e.set_pipeline(project.build_pipeline(p), 'phase2_pipeline')
    return e.pipeline_version


if 'phase2' in project.list_experimenters():
    e = project.load_experimenter('phase2', df_train)
else:
    e = project.experimenter(
        'phase2', df_train,
        sp=StratifiedShuffleSplit(n_splits=1, random_state=123, train_size=0.8),
        sp_v=StratifiedShuffleSplit(n_splits=1, random_state=123, train_size=0.9),
        splitter_params={'y': y},
        pipeline_name='phase2_pipeline',
        pipeline_version=project.build_pipeline(p).version,
    )
e.pipeline_version

17

In [14]:
collectors = e.collectors


def conn(**kw):
    return {'__ref__': 'mllabs.Connector', '__params__': kw}


MA = 'mllabs.collector.ModelAttrCollector'
collectors.set_collector('xgb_evals_results', MA, conn(processor='xgboost.XGBClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('lgb_evals_results', MA, conn(processor='lightgbm.LGBMClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('cb_evals_results', MA, conn(processor='catboost.CatBoostClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector('nn_evals', MA, conn(processor='mllabs.nn.NNClassifier'), params={'result_key': 'evals_result'})
collectors.set_collector(
    'bAcc', 'mllabs.collector.MetricCollector', conn(edges=y_edges),
    params={'output_var': '-1:',
            'metric_func': {'__callable__': 'sklearn.metrics.balanced_accuracy_score'},
            'include_train': True})
collectors.set_collector('lgb_feature_importance', MA, conn(processor='lightgbm.LGBMClassifier', edges=y_edges), params={'result_key': 'feature_importances'})
collectors.set_collector('xgb_feature_importance_gain', MA, conn(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'gain'}})
collectors.set_collector('xgb_feature_importance_cover', MA, conn(processor='xgboost.XGBClassifier', edges=y_edges), params={'result_key': 'feature_importances', 'params': {'importance_type': 'cover'}})
collectors.set_collector('cb_feature_importance', MA, conn(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_pvc'})
collectors.set_collector('cb_interaction', MA, conn(processor='catboost.CatBoostClassifier', edges=y_edges), params={'result_key': 'feature_importances_interaction'})
collectors.set_collector('lr_coef', MA, conn(processor='sklearn.linear_model.LogisticRegression', edges=y_edges), params={'result_key': 'coef'})

collectors.names()

I0000 00:00:1785854549.670709   75715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785854549.711793   75715 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785854550.517032   75715 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will

['xgb_evals_results',
 'lgb_evals_results',
 'nn_evals',
 'bAcc',
 'lgb_feature_importance',
 'xgb_feature_importance_gain',
 'xgb_feature_importance_cover',
 'cb_feature_importance',
 'cb_interaction',
 'lr_coef',
 'cb_evals_results',
 'stack_oof']

`set_collector`는 **등록 즉시 영속화**된다(`exp/phase2/collectors/collectors.db` + `__params/{name}.pkl`).
`Collectors.save()`는 없고, 다음 세션에서는 `e.collectors` 접근 자체가 복원이다 —
레지스트리는 이 run의 것이라 `load_experimenter('phase2', ...)`가 Collector 정의·데이터·이력을 같이 되살린다.

레지스트리가 run 소유인 이유: Collector가 쓰는 건 전부 노드 이름만으로 키잉된다
(`MetricCollector` PK `(node, idx, inner_idx, split)`, 파일 기반은 `{path}/{node}...`).
프로젝트 전역이었다면 이름이 겹치는 Trial마다 두 run이 서로의 결과를 덮어썼다.

`Connector(role='head')`의 `role`도 없어졌다 — Collector는 애초에 Trial job에만 붙고 노드 job엔 안 붙어서
걸러야 할 대상이 없었다.

In [16]:
def folds(trials):
    return [(t, o, i) for t in trials
            for o in range(e.get_n_splits())
            for i in range(e.get_n_splits_inner())]


def run(trials, **kw):
    """collectors 인자를 안 주면 이 run에 등록된 Collector 전부가 붙고,
    수집 이력도 e.collectors.hist에 남는다."""
    kw = {'n_jobs': 2, 'gpu_id_list': [0], 'logger': logger, **kw}
    e.exp(folds(trials), project.trials, **kw)

In [17]:
display(Markdown(p.desc_node('tgt_km3000')))
display(Markdown(p.desc_pipeline()))

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_tgt_km3000["pre_ft2/tgt_km3000"]
        tgt_km3000_dummy[ ]
        style tgt_km3000_dummy fill:none,stroke:none
    end
    style node_tgt_km3000 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    DataSource -->|X,y| node_tgt_km3000
```

**Path from DataSource to 'pre_ft2/tgt_km3000' (1 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | Data Source | `{km3000}` |
| y | Data Source | `{class}` |

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_pre["pre"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_coov["coov"]
        style node_coov fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_delta_100["delta_100"]
        style node_delta_100 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_delta_500["delta_500"]
        style node_delta_500 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_alpha_100["alpha_100"]
        style node_alpha_100 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_alpha_500["alpha_500"]
        style node_alpha_500 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_cat0["cat0"]
        style node_cat0 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_cat1["cat1"]
        style node_cat1 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe_coov["ohe_coov"]
        style node_ohe_coov fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft2["pre_ft2"]
        node_tgt_km3000["tgt_km3000"]
        style node_tgt_km3000 fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_tgt_delta["tgt_delta"]
        style node_tgt_delta fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_tgt_alpha["tgt_alpha"]
        style node_tgt_alpha fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_lda_mags["lda_mags"]
        style node_lda_mags fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre_ft2 fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp___datasource__["__datasource__"]
    end
    style grp___datasource__ fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_pre_ft["pre_ft"]
    end
    style grp_pre_ft fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_pre
    DataSource --> grp_pre_ft2
    grp_pre --> grp_pre_ft2
```

In [18]:
with e.os_log():
    e.build()

No stage nodes to build


## Round 1 — 기본 피처

In [19]:
round1 = round_trials(1, xgb_cols=X_num)
with e.os_log():
    run([t for t in round1 if t.name == 'xgb1'])

No trials to run


In [20]:
with e.os_log():
    run(round1)

No trials to run


## Round 2 — `tgt_km3000` 투입

노드는 그대로라 파이프라인을 다시 빌드할 필요가 없다. Trial만 새로 만들어 돌린다.

In [21]:
round2 = round_trials(2, 'tgt_km3000')
with e.os_log():
    run(round2)

No trials to run


In [22]:
collectors.get_collector('lgb_feature_importance').get_attrs_agg('lgb2').sort_values(ascending=False).iloc[:10]

delta                        651.0
redshift                     572.0
redshift_log                 537.0
alpha90                      480.0
g_z                          363.0
tgt_km3000__km3000_GALAXY    363.0
tgt_km3000__km3000_STAR      335.0
g_log                        317.0
u_g                          313.0
r_log                        275.0
dtype: float64

In [23]:
collectors.get_collector('xgb_feature_importance_gain').get_attrs_agg('xgb2').sort_values(ascending=False).iloc[:10]

g_z                          682.720154
redshift                     393.362610
u_i                          184.193619
redshift_1e-4                178.758728
z_log                        178.002701
mag_std                      155.960892
g_log                         94.202972
tgt_km3000__km3000_GALAXY     85.266930
tgt_km3000__km3000_STAR       84.644470
g_i                           71.461990
dtype: float64

In [24]:
collectors.get_collector('lr_coef').get_attrs_agg('lr2').sort_values(ascending=False).iloc[:10]

2  std__u_log                 6.390728
1  std__r_log                 5.988768
   std__z_log                 5.709977
   std__i_log                 5.441443
   std__g_log                 4.120085
0  std__z_log                 3.433065
2  std__z                     2.378478
0  tgt_km3000__km3000_STAR    2.357097
2  std__i                     2.303545
   std__r                     2.118813
dtype: float64

In [25]:
collectors.get_collector('cb_interaction').get_attrs_agg('cb2').sort_values(ascending=False).iloc[:20]

feat1          feat2                    
redshift_log   g_z                          4.963188
redshift       g_z                          4.756604
redshift_log   tgt_km3000__km3000_GALAXY    4.751363
redshift       tgt_km3000__km3000_GALAXY    4.641409
               redshift_log                 4.141631
redshift_log   redshift_1e-4                4.090602
g_z            tgt_km3000__km3000_GALAXY    3.426295
redshift       redshift_1e-4                3.275893
redshift_1e-4  g_z                          3.090055
               tgt_km3000__km3000_GALAXY    2.625070
redshift_log   z_log                        2.458256
redshift       z_log                        2.004085
redshift_log   tgt_km3000__km3000_STAR      1.890039
redshift       tgt_km3000__km3000_STAR      1.866339
               mag_std                      1.726187
g_z            z_log                        1.565888
redshift_log   mag_std                      1.542945
mag_std        tgt_km3000__km3000_GALAXY    1.513337
g_z  

## Round 3 — `delta` 타겟 인코딩

여기서부터는 노드가 추가되므로 `adopt()`(= `build_pipeline` + `set_pipeline`)를 거쳐야 한다.
Pipeline은 불변이라 builder 수정이 진행 중인 run에 새어 들어가지 않는다.

In [26]:
with e.os_log():
    p.set_node('delta_100', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['delta'])}, params={'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('delta_500', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['delta'])}, params={'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_delta', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': 'delta_100:(*) + delta_500:(*)'}, params={'target_type': 'multiclass'})
    adopt()
    e.build()

No stage nodes to build


In [27]:
round3 = round_trials(3, 'tgt_km3000', 'tgt_delta')
with e.os_log():
    run(round3)

No trials to run


## Round 4 — `alpha90` 타겟 인코딩

In [28]:
with e.os_log():
    p.set_node('alpha_100', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['alpha90'])}, params={'n_bins': 100, 'encode': 'ordinal'})
    p.set_node('alpha_500', grp='pre', processor='sklearn.preprocessing.KBinsDiscretizer', edges={'X': dsl_set(['alpha90'])}, params={'n_bins': 500, 'encode': 'ordinal'})
    p.set_node('tgt_alpha', grp='pre_ft2', processor='sklearn.preprocessing.TargetEncoder', edges={'X': 'alpha_100:(*) + alpha_500:(*)'}, params={'target_type': 'multiclass'})
    adopt()
    e.build()

No stage nodes to build


In [29]:
round4 = round_trials(4, 'tgt_km3000', 'tgt_delta', 'tgt_alpha')
with e.os_log():
    run(round4)

No trials to run


In [30]:
collectors.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending=False).iloc[:15]

,test,train,valid
cb4,0.957820,0.962887,0.957953
cb3,0.957690,0.963358,0.957602
lgb5,0.957674,0.961794,0.957184
cb5,0.957563,0.962591,0.957747
cb2,0.957378,0.962238,0.957516
lgb4,0.957304,0.961327,0.956940
lgb3,0.957216,0.961562,0.956855
lgb2,0.956519,0.960521,0.956464
xgb5,0.954914,0.974904,0.954743
xgb3,0.954468,0.973918,0.954389


## Round 5 — LDA + 범주형 좌표 bin

In [31]:
with e.os_log():
    p.set_node('cat0', grp='pre', processor='mllabs.processor.CatConverter', edges={'X': 'alpha_500:(*) + delta_500:(*)'})
    p.set_node('cat1', grp='pre', processor='mllabs.processor.CatConverter', edges={'X': 'alpha_100:(*) + delta_100:(*)'})
    p.set_node('lda_mags', grp='pre_ft2', processor='sklearn.discriminant_analysis.LinearDiscriminantAnalysis', edges={'X': dsl_set(X_mags)})
    adopt()
    e.build()

No stage nodes to build


In [32]:
EXTRA5 = ('tgt_km3000', 'tgt_delta', 'tgt_alpha', 'lda_mags')
round5 = round_trials(5, *EXTRA5)

nn_base = edges_for(*EXTRA5)['nn']
cat_cols = {'__ref__': 'mllabs.ColSelector', '__params__': {'dsl_string': '*@categorical'}}
round5 += [
    trial('nn6', 'nn', nn_base + ' + ' + dsl_set(X_nom2) + ' + cat0:(*)', cat_cols=cat_cols),
    trial('nn7', 'nn', nn_base + ' + ' + dsl_set(X_nom2) + ' + cat1:(*)', cat_cols=cat_cols),
    trial('nn8', 'nn', nn_base + ' + ' + dsl_set(X_nom2), cat_cols=cat_cols),
    trial('nn9', 'nn', nn_base + ' + cat1:(*)', cat_cols=cat_cols),
]

with e.os_log():
    run(round5)

No trials to run


In [33]:
collectors.get_collector('bAcc').get_metrics_agg(None)[0].sort_values('test', ascending=False)

,test,train,valid
cb4,0.957820,0.962887,0.957953
cb3,0.957690,0.963358,0.957602
lgb5,0.957674,0.961794,0.957184
cb5,0.957563,0.962591,0.957747
cb2,0.957378,0.962238,0.957516
lgb4,0.957304,0.961327,0.956940
lgb3,0.957216,0.961562,0.956855
lgb2,0.956519,0.960521,0.956464
xgb5,0.954914,0.974904,0.954743
xgb3,0.954468,0.973918,0.954389


## dtype 기반 selector (`@numeric` / `@categorical` / `@binary` / `@float` / `@int` / `@string`)

edges DSL에서 `@numeric`/`@categorical` 등은 실제 컬럼의 dtype을 보고 선택하는 selector다 (processor 불필요).
DataSource 최상위(`*@numeric`)에 바로 걸면 `id`/`class_i`(target)/`sample_weight`처럼 스키마에 없는 raw 컬럼까지
딸려 들어올 수 있어 위험하므로, 이미 확정된 **노드 출력 namespace 안에서만** 사용한다.

In [34]:
with e.os_log():
    p.set_node('ohe_coov', grp='pre', processor='sklearn.preprocessing.OneHotEncoder', edges={'X': 'coov:(*@categorical)'}, 
               params={'sparse_output': False, 'handle_unknown': 'ignore'})
    adopt()
    e.build()

No stage nodes to build


In [35]:
dt1 = trial('dt1', 'dt', 'std:(*@numeric) + ohe_coov:(* - ^.*__km3000.*$) + tgt_km3000:(*)')
run([dt1])
collectors.get_collector('bAcc').get_metrics_agg('dt1')[0]

No trials to run


,test,train,valid
dt1,0.926545,1.0,0.926051


## 수집 이력 — `CollectHist`

`collectors.hist`는 `(collector, node, outer_idx, inner_idx)`마다 한 행을 남긴다.
`experimenter` 키는 없다 — hist가 이 run의 레지스트리 안에 있어서, 어느 run이냐는 db가 어디
있느냐로 답한다(`node_hist`에 `run_name`이 없는 것과 같다).

- `status`: `'collected'` / `'empty'`(예외 없이 `None` 반환 — 보통 `output_var` 설정 실수) / `'error'`
- `info`: 에러일 때 `{phase, type, message, traceback}`. `phase`는 `'output'`/`'ext'`/`'collect'`/`'push'`
- `elapsed`: `collect()` 호출 시간

예전엔 수집 실패가 `collector.warnings`(메모리)에만 쌓였고, 멀티워커에선 워커 사본에 쌓였다가 그대로
버려졌다. 지금은 항상 부모 프로세스가 기록한다.

In [36]:
hist = collectors.hist
pd.DataFrame(hist.get_hist()).groupby(['collector_name', 'status']).size().unstack(fill_value=0)

status,collected
collector_name,
bAcc,30
cb_evals_results,5
cb_feature_importance,5
cb_interaction,5
lgb_evals_results,5
lgb_feature_importance,5
lr_coef,5
nn_evals,9
stack_oof,15


In [37]:
for row in hist.get_hist(status='error'):
    print(row['collector_name'], row['node_name'], (row['outer_idx'], row['inner_idx']),
          row['info']['phase'], row['info']['type'], row['info']['message'])

In [38]:
pd.DataFrame(hist.get_hist()).groupby('collector_name')['elapsed'].agg(['sum', 'mean', 'max']).sort_values('sum', ascending=False)

,sum,mean,max
collector_name,,,
bAcc,0.550702,0.018357,0.040116
cb_interaction,0.067959,0.013592,0.022048
nn_evals,0.059662,0.006629,0.011990
cb_feature_importance,0.052093,0.010419,0.016394
cb_evals_results,0.033614,0.006723,0.008782
lr_coef,0.025226,0.005045,0.011356
xgb_evals_results,0.024432,0.004886,0.005951
lgb_evals_results,0.014034,0.002807,0.003858
xgb_feature_importance_gain,0.003509,0.000702,0.000870


### 나중에 붙인 Collector가 아무것도 못 보는 경우

수집은 Trial job이 실행될 때의 부수효과다. `TrialStore.experiment_hist`에 이미 `'built'`로 기록된 fold는
`exp()`가 스킵하므로, 실험이 끝난 뒤에 Collector를 새로 붙이고 `exp()`를 다시 불러봐야 아무것도 수집되지
않는다. `CollectHist`를 조회하면 그 사실이 드러난다.

다시 수집하려면 그 Trial의 fold 이력을 명시적으로 지워야 한다 — `reset_nodes()`는 아티팩트만 지우고
스킵 판정에 쓰이는 이력은 건드리지 않는다.

In [39]:
collectors.set_collector('cb_evals_results', MA, conn(processor='catboost.CatBoostClassifier'),
                         params={'result_key': 'evals_result'}, exist='replace')

cb_trials = [t for t in round1 + round2 + round3 + round4 + round5 if t.name.startswith('cb')]
missing = [t for t in cb_trials
           if not hist.get_hist(collector_name='cb_evals_results', node_name=t.name)]
[t.name for t in missing]

[]

In [40]:
for t in missing:
    project.trials.remove_hist(trial_name=t.name, experimenter=e.name)
e.reset_nodes([t.name for t in missing])

with e.os_log():
    run(missing)

No trials to run


## 상태 점검

Trial 실행 이력은 `TrialStore`(프로젝트 전역)에, 노드 이력은 이 run의 `NodeStore`에 있다.

In [41]:
e.show_error_nodes(trial_store=project.trials)
display(Markdown(e.get_node_info()))

# Experiment Pipeline Summary

- **DataSource**

## std
- **Processor**: sklearn.preprocessing.StandardScaler
- **Method**: transform
- **Edges**: X: {redshift_log, lof, u, g, r, i, z, mag_mean, mag_std, mag_min, mag_max, mag_range, u_log, g_log, r_log, i_log, z_log, u_g, u_r, u_i, u_z, g_r, g_i, g_z, r_i, r_z, i_z}

## ohe
- **Processor**: sklearn.preprocessing.OneHotEncoder
- **Method**: transform
- **Edges**: X: {spectral_type, spectral_type_galaxy_population}

## coov
- **Processor**: mllabs.processor.CatOOVFilter
- **Method**: transform
- **Edges**: X: {galaxy_population, spectral_type, spectral_type_galaxy_population, km3000}
- **Descendants**: ['ohe_coov']

## tgt_km3000
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: {km3000}

## delta_100
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {delta}
- **Descendants**: ['cat1', 'tgt_delta']

## delta_500
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {delta}
- **Descendants**: ['cat0', 'tgt_delta']

## tgt_delta
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: delta_100:(*) + delta_500:(*)

## alpha_100
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {alpha90}
- **Descendants**: ['cat1', 'tgt_alpha']

## alpha_500
- **Processor**: sklearn.preprocessing.KBinsDiscretizer
- **Method**: transform
- **Edges**: X: {alpha90}
- **Descendants**: ['cat0', 'tgt_alpha']

## tgt_alpha
- **Processor**: sklearn.preprocessing.TargetEncoder
- **Method**: fit_transform
- **Edges**: y: {class}, X: alpha_100:(*) + alpha_500:(*)

## cat0
- **Processor**: mllabs.processor.CatConverter
- **Method**: transform
- **Edges**: X: alpha_500:(*) + delta_500:(*)

## cat1
- **Processor**: mllabs.processor.CatConverter
- **Method**: transform
- **Edges**: X: alpha_100:(*) + delta_100:(*)

## lda_mags
- **Processor**: sklearn.discriminant_analysis.LinearDiscriminantAnalysis
- **Method**: fit_transform
- **Edges**: y: {class}, X: {u, g, r, i, z}

## ohe_coov
- **Processor**: sklearn.preprocessing.OneHotEncoder
- **Method**: transform
- **Edges**: X: coov:(*@categorical)


In [42]:
pd.DataFrame(project.trials.get_hist(experimenter=e.name))[
    ['trial_name', 'outer_idx', 'inner_idx', 'pipeline_version', 'status']
]

,trial_name,outer_idx,inner_idx,pipeline_version,status
0,cb1,0,0,1,built
1,cb2,0,0,1,built
2,cb3,0,0,2,built
3,cb4,0,0,3,built
4,cb5,0,0,4,built
5,dt1,0,0,5,built
6,lgb1,0,0,1,built
7,lgb2,0,0,1,built
8,lgb3,0,0,2,built
9,lgb4,0,0,3,built


## Early stopping 지점 측정

`evals_result` collector가 fold마다 남긴 학습 곡선에서 **실제로 몇 라운드에 멈췄는지**를 뽑는다.
곡선의 인덱스는 `(iteration, metric, split)` 3단이고, validation split은 eval_set 순서상 항상
마지막이다 — xgb `validation_1` / lgb `valid_1` / cb `validation` / nn `valid`. 이름이 제각각이라
`split=`을 안 주면 정렬 순 마지막을 validation으로 본다.

- `rounds`: 기록된 iteration 수 = 학습이 멈춘 지점 (NN은 epoch 단위)
- `best_iter`: validation 기준 최적 iteration
- `after_best`: `rounds - 1 - best_iter`. `early_stopping_rounds`와 같으면 정상 발동,
  그보다 작은데 `rounds`가 `n_estimators`(10000)에 붙어 있으면 **멈추기 전에 상한에 닿은 것**

cb는 `eval_metric='AUC'`를 validation에만 기록하고 loss(`MultiClass`)는 learn/validation 양쪽에
기록하므로, metric마다 행이 따로 나온다. 실제 정지를 결정한 건 `eval_metric` 쪽이다.


In [43]:
import re

MAXIMIZE = re.compile(r'auc|acc|f1|precision|recall|r2|map|ndcg', re.I)


def stop_rounds(names=None, split=None):
    rows = []
    for cname in names or [n for n in collectors.names() if 'eval' in n]:
        for node, outers in collectors.get_collector(cname).get_attrs().items():
            for oi, inners in enumerate(outers or []):
                for ii, s in enumerate(inners):
                    if s is None or len(s) == 0:
                        continue
                    sp = split or sorted(s.index.get_level_values(2).unique())[-1]
                    curve = s.xs(sp, level=2)
                    for metric in curve.index.get_level_values(1).unique():
                        m = curve.xs(metric, level=1)
                        best = int(m.idxmax() if MAXIMIZE.search(metric) else m.idxmin())
                        last = int(m.index.max())
                        rows.append({'collector': cname, 'node': node,
                                     'outer_idx': oi, 'inner_idx': ii, 'split': sp,
                                     'metric': metric, 'rounds': last + 1,
                                     'best_iter': best, 'best_score': m.loc[best],
                                     'after_best': last - best})
    return pd.DataFrame(rows)


es = stop_rounds()
es.sort_values(['node', 'metric']).reset_index(drop=True)


,collector,node,outer_idx,inner_idx,split,metric,rounds,best_iter,best_score,after_best
0,cb_evals_results,cb1,0,0,validation,AUC:type=Mu,2111,2060,0.997579,50
1,cb_evals_results,cb1,0,0,validation,MultiClass,2111,2081,0.094181,29
2,cb_evals_results,cb2,0,0,validation,AUC:type=Mu,1880,1829,0.997945,50
3,cb_evals_results,cb2,0,0,validation,MultiClass,1880,1879,0.086571,0
4,cb_evals_results,cb3,0,0,validation,AUC:type=Mu,2093,2042,0.997972,50
5,cb_evals_results,cb3,0,0,validation,MultiClass,2093,2090,0.085909,2
6,cb_evals_results,cb4,0,0,validation,AUC:type=Mu,1845,1794,0.997979,50
7,cb_evals_results,cb4,0,0,validation,MultiClass,1845,1844,0.085673,0
8,cb_evals_results,cb5,0,0,validation,AUC:type=Mu,1639,1588,0.997967,50
9,cb_evals_results,cb5,0,0,validation,MultiClass,1639,1634,0.085977,4


In [44]:
es.groupby(['collector', 'metric'])[['rounds', 'best_iter', 'after_best']].agg(['mean', 'max']).round(1)

rounds       best_iter  \
                                                     mean   max      mean   
collector         metric                                                    
cb_evals_results  AUC:type=Mu                      1913.6  2111    1862.6   
                  MultiClass                       1913.6  2111    1905.6   
lgb_evals_results multi_logloss                     136.0   147      85.0   
nn_evals          loss                               33.3    62      22.3   
                  sparse_categorical_crossentropy    33.3    62      22.3   
xgb_evals_results mlogloss                           95.0   110      44.0   

                                                        after_best      
                                                    max       mean max  
collector         metric                                                
cb_evals_results  AUC:type=Mu                      2060       50.0  50  
                  MultiClass                       2090        7.0  29  
lgb_evals_results multi_logloss                      96       50.0  50  
nn_evals          loss                               51       10.0  10  
                  sparse_categorical_crossentropy    51       10.0  10  
xgb_evals_results mlogloss                           59       50.0  50

Trainer는 validation set 없이 전체 데이터로 학습하므로 early stopping을 걸 수 없다 —
위 `best_iter`를 `n_estimators`로 **고정**해서 넘겨야 한다. 이 실험의 inner train은 전체의
`0.8 * 0.9 = 0.72`이므로, 데이터가 늘어난 만큼 보정하려면 `best_iter / 0.72` 정도가 출발점이다.


## Stacking — OOF 메타피처와 같은 shape의 Inferencer

앞의 실험은 outer가 1-fold(`ShuffleSplit`)라 held-out이 전체의 20%뿐이었다. 스태킹 메타피처는
**모든 행에 대한 out-of-fold 예측**이 필요하므로 outer를 5-fold `StratifiedKFold`로 바꾼 실험을
따로 만든다 — `StackingCollector._build_index`가 outer fold들의 `test_idx`를 이어붙이는데,
5-fold면 그 합집합이 전체 행을 정확히 한 번씩 덮는다.

`sp_v=None`(inner 분할 없음)로 두는 이유는 early stopping을 쓰지 않기 때문이고, 사실 이 둘은
서로를 강제한다 — validation set이 없으면 xgb `early_stopping_rounds`, lgb `early_stopping`,
nn EarlyStopping이 **전부 에러**를 낸다. 그래서 라운드 수를 앞 절에서 측정한 eval 커브로
고정한다.


In [45]:
BASE = ('lgb', 'xgb', 'nn')
ALL_TRIALS = {t.name: t for t in round1 + round2 + round3 + round4 + round5}


def family(name):
    return name.rstrip('0123456789')


score = collectors.get_collector('bAcc').get_metrics_agg(None)[0]
cand = score.loc[[n for n in score.index if n in ALL_TRIALS and family(n) in BASE]]

TRAIN_FRAC_PHASE2 = 0.8 * 0.9   # outer 0.8 x inner 0.9
TRAIN_FRAC_STACK = 4 / 5        # 5-fold, inner 없음

# 계열별 최고 trial과, 그 trial 자신의 eval 커브에서 나온 라운드
picked = {}
for m in BASE:
    rows = cand[cand.index.map(family) == m]
    name = rows['test'].idxmax()
    # metric이 여럿이면 가장 늦게까지 개선된 쪽을 취한다 (보수적)
    best_iter = int(es.loc[es['node'] == name, 'best_iter'].max())
    picked[m] = (name, round((best_iter + 1) * TRAIN_FRAC_STACK / TRAIN_FRAC_PHASE2))

pd.DataFrame(picked, index=['trial', 'n_rounds']).T


,trial,n_rounds
lgb,lgb5,93
xgb,xgb5,48
nn,nn9,18


In [46]:
NO_ES = ('early_stopping', 'early_stopping_rounds', 'eval_metric')


def stack_trial(name, src, model, n_rounds):
    """src를 그대로 복사하되 early stopping을 빼고 라운드를 고정한 predict_proba trial.

    MODELS[model]이 아니라 *뽑힌 trial 자신의* params/edges/adapter에서 출발한다 —
    nn6~nn9처럼 계열 기본값 위에 cat_cols 같은 개별 인자가 얹힌 경우가 있어서다.
    """
    params = {k: v for k, v in src.params.items() if k not in NO_ES}
    params['epochs' if model == 'nn' else 'n_estimators'] = n_rounds
    return Trial(name, src.processor, dict(src.edges), method='predict_proba',
                 adapter=src.adapter, params=params)


# 이름을 새로 준다 — TrialStore는 이름이 PK이고 프로젝트 전역이라,
# 'lgb5'를 재사용하면 phase2의 그 정의를 덮어쓴다.
stk_trials = [stack_trial(f'{name}_stk', ALL_TRIALS[name], m, n)
              for m, (name, n) in picked.items()]
[(t.name, t.params.get('n_estimators', t.params.get('epochs'))) for t in stk_trials]


[('lgb5_stk', 93), ('xgb5_stk', 48), ('nn9_stk', 18)]

In [47]:
from sklearn.model_selection import StratifiedKFold

stk_pipeline = project.load_pipeline('phase2_pipeline')

if 'stack' in project.list_experimenters():
    e_stk = project.load_experimenter('stack', df_train)
else:
    e_stk = project.experimenter(
        'stack', df_train,
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=123),
        sp_v=None,
        splitter_params={'y': y},
        pipeline_name='phase2_pipeline',
        pipeline_version=stk_pipeline.version,
    )

with e_stk.os_log():
    e_stk.build(n_jobs=2, logger=logger)

e_stk.get_n_splits(), e_stk.get_n_splits_inner()


No stage nodes to build


(5, 1)

`output_var='1:'`가 "한 클래스 확률 제외"다. DSL의 slice는 그 노드 **자기 출력 컬럼**에 대한
위치 슬라이스라(`eval_expr`의 `list(columns[node])`) 클래스가 몇 개든 첫 컬럼 하나만 떨어진다.
predict_proba 출력 컬럼은 `{노드}__{y}_{class}` 형식이므로 여기선 `..._class_i_0`이 빠지고
`_1`/`_2`만 남는다 — 확률의 합이 1이라 한 열은 나머지로 결정된다.

Connector에 `edges=y_edges`를 주는 건 매칭용만이 아니다. `get_dataset(include_target=True)`이
`connector.edges['y']`를 읽어 target 컬럼을 붙이므로, 이게 없으면 피처만 나온다.

`stack_oof`는 `e_stk`의 레지스트리에 등록한다 — 레지스트리는 run 소유라 `e`의 것과 별개이고,
그래서 같은 이름의 Trial을 두 run이 돌려도 결과가 서로 덮이지 않는다.

`collectors=['stack_oof']`로 이름을 하나만 주는 건 이 run에 다른 Collector가 붙는 걸 막기
위해서가 아니라(여긴 이것뿐이다) 의도를 명시하기 위해서다. 참고로 `bAcc`를 여기 등록했다면
걸렸을 것이다 — `output_var='-1:'` + `balanced_accuracy_score` 조합이라 `method='predict'`
출력(라벨 한 열)을 전제하는데, predict_proba 출력에 걸리면 확률을 라벨로 채점하려다 에러가 난다.
수집 이력은 선택과 무관하게 언제나 그 run의 `collectors.hist`에 남는다.

한 가지 주의: `StackingCollector`는 outer fold 결과를 메모리(`_outer_buf`)에 모았다가 5개가 다
차면 그때서야 `{node}.pkl`로 쓴다. 즉 **5 fold가 한 번의 `exp()` 호출에서 다 돌아야** 저장된다 —
중간에 실패해 일부 fold만 `'built'`로 기록되면 다음 호출에서 그 fold들은 스킵되고, 버퍼가 영영
안 차서 파일이 안 생긴다. 그럴 땐 `project.trials.remove_hist(trial_name=..., experimenter=...)`로
fold 이력을 지우고 전부 다시 돌려야 한다.


In [48]:
e_stk.collectors.set_collector(
    'stack_oof', 'mllabs.collector.StackingCollector',
    conn(node_query=[t.name for t in stk_trials], edges=y_edges),
    params={'output_var': '1:', 'method': 'mean'}, exist='replace')

folds_stk = [(t, o, i) for t in stk_trials
             for o in range(e_stk.get_n_splits())
             for i in range(e_stk.get_n_splits_inner())]

with e_stk.os_log():
    e_stk.exp(folds_stk, project.trials,
              collectors=['stack_oof'],
              n_jobs=2, gpu_id_list=[0], logger=logger)


No trials to run


In [49]:
ds = e_stk.collectors.get_collector('stack_oof').get_dataset(e_stk)
print(ds.shape, ds.columns)
ds.head()


(577347, 7) ['nn9_stk__class_i_1', 'nn9_stk__class_i_2', 'lgb5_stk__class_i_1', 'lgb5_stk__class_i_2', 'xgb5_stk__class_i_1', 'xgb5_stk__class_i_2', 'class_i']


nn9_stk__class_i_1,nn9_stk__class_i_2,lgb5_stk__class_i_1,lgb5_stk__class_i_2,xgb5_stk__class_i_1,xgb5_stk__class_i_2,class_i
f64,f64,f64,f64,f64,f64,i8
0.000266,0.999733,0.000329,0.999652,0.00071,0.999263,2
0.001831,0.00095,0.001658,0.010129,0.003451,0.003088,0
0.00042,0.999576,0.000673,0.999297,0.000961,0.998984,2
0.999964,0.000036,0.999866,0.000121,0.998996,0.000988,1
0.000036,0.999964,0.000311,0.999664,0.000087,0.999899,2


### Trainer — 같은 정의를 전체 데이터로

`Predictor.from_trial(t, experimenter=...)`은 실행 정의를 그대로 복사하고 **이름도 그대로**
가져간다(출처는 `src_trial`/`src_experimenter`에 따로 남는다). 이게 shape 일치의 근거다 —
predict_proba 출력 컬럼명이 `{노드 이름}__{y}_{class}`라서, Predictor 이름이 Trial 이름과 같으면
Inferencer가 내놓는 컬럼이 OOF 데이터셋의 컬럼과 문자 그대로 같아진다. `v='1:'`로 같은 슬라이스를
걸면 target을 뺀 나머지가 정확히 대응한다.

Trainer에 splitter를 안 주면 전체 데이터 단일 fold로 학습한다(`n_splits=1`). 라운드 수는 OOF
쪽과 같은 값을 그대로 쓴다 — 학습 데이터가 0.8에서 1.0으로 늘어난 만큼 더 태울 여지는 있지만,
그러면 `from_trial`이 복사한 정의를 손대야 해서 출처 기록의 의미가 흐려진다.

**컬럼 순서는 이름으로 맞춰야 한다.** 베이스 모델이 여럿이면 두 쪽의 노드 순서가 다를 수 있다 —
`get_dataset`은 `_get_saved_nodes()`가 `path.glob('*.pkl')`로 훑은 파일시스템 순서를 따르고,
`Inferencer.process`는 `selected_predictors`(= `predictor_names()`) 순서를 따른다. 집합은 같아도
위치는 보장되지 않으므로 아래 셀은 이름으로 대조한 뒤 `select(feat_cols)`로 순서를 맞춘다.


In [50]:
from mllabs import Predictor

preds = [Predictor.from_trial(t, experimenter=e_stk.name) for t in stk_trials]

if 'stack_final' in project.list_trainers():
    t_stk = project.load_trainer('stack_final', df_train)
else:
    t_stk = project.trainer('stack_final', df_train,
                            pipeline_name='phase2_pipeline',
                            pipeline_version=stk_pipeline.version)

t_stk.train(preds, n_jobs=1, gpu_id_list=[0], logger=logger)
t_stk.predictor_names(), t_stk.get_n_splits()


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

W0000 00:00:1785854583.615481   75715 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Train complete: 15 node(s)


(['lgb5_stk', 'xgb5_stk', 'nn9_stk'], 1)

In [52]:
inf = t_stk.to_inferencer(v='1:')
test_meta = inf.process(df_test)

prefixes = tuple(f'{n}__' for n in t_stk.predictor_names())
feat_cols = [c for c in ds.columns if c.startswith(prefixes)]

print('OOF :', feat_cols)
print('test:', list(test_meta.columns))
assert sorted(feat_cols) == sorted(test_meta.columns)

test_meta = test_meta.select(feat_cols)   # 순서까지 맞춘다 (아래 주의 참고)
print(ds.shape, test_meta.shape)
test_meta.head()


OOF : ['nn9_stk__class_i_1', 'nn9_stk__class_i_2', 'lgb5_stk__class_i_1', 'lgb5_stk__class_i_2', 'xgb5_stk__class_i_1', 'xgb5_stk__class_i_2']
test: ['lgb5_stk__class_i_1', 'lgb5_stk__class_i_2', 'xgb5_stk__class_i_1', 'xgb5_stk__class_i_2', 'nn9_stk__class_i_1', 'nn9_stk__class_i_2']
(577347, 7) (247435, 6)


nn9_stk__class_i_1,nn9_stk__class_i_2,lgb5_stk__class_i_1,lgb5_stk__class_i_2,xgb5_stk__class_i_1,xgb5_stk__class_i_2
f32,f32,f64,f64,f32,f32
0.003637,0.995774,0.001949,0.997975,0.000404,0.999482
0.000878,0.999122,0.002608,0.997334,0.001336,0.998621
0.000076,0.999869,0.000239,0.998108,0.000311,0.99577
0.001959,0.001953,0.002075,0.005513,0.001332,0.002215
0.000153,0.999847,0.000548,0.999425,0.000802,0.999159


In [58]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

clf_meta = LogisticRegression(solver = 'lbfgs')
X_meta = [i for i in ds.columns if i != 'class_i']
skf = StratifiedKFold(5, random_state = 123, shuffle = True)
scores = cross_val_score(
    clf_meta, ds[X_meta], ds['class_i'], scoring = 'balanced_accuracy', cv =skf
)
scores.mean()

np.float64(0.9569291991038986)

In [60]:
prd = clf_meta.fit(ds[X_meta], ds['class_i']).predict(test_meta)

In [61]:
pd.Series(prd).value_counts()

2    162435
1     49843
0     35157
Name: count, dtype: int64